# Project Sentinel — Eyes of the Highway Reserve
### ACM BPHC GenAI Induction 2026 — Task 2

**Goal:** Build and compare land-cover classifiers on the EuroSAT (RGB) dataset as a stand-in for a wildlife-reserve
satellite feed. We train/compare four models:

1. CNN from scratch (TinyVGG) — **no augmentation**
2. CNN from scratch (TinyVGG) — **with augmentation**
3. Fine-tuned pretrained model (ResNet18) — **no augmentation**
4. Fine-tuned pretrained model (ResNet18) — **with augmentation**

Each is evaluated on loss/accuracy curves, final test accuracy, and a confusion matrix. The final section
assembles the "field report" comparing all four.

> Run this notebook top to bottom in Google Colab (GPU runtime recommended: Runtime → Change runtime type → T4 GPU)
> or locally with a CUDA-capable GPU. On CPU it will still run, just slower — reduce `EPOCHS` if needed.


## 0. Setup

In [ ]:
# If running in Colab, uncomment the line below
# !pip install torch torchvision torchsummary scikit-learn seaborn -q

import os
import time
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 1. Dataset — EuroSAT (RGB)

`torchvision.datasets.EuroSAT` downloads and extracts the RGB/JPEG version of EuroSAT automatically
(10 classes: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop,
Residential, River, SeaLake — a solid proxy for forest / river / highway / farmland / residential tagging).

If `download=True` fails in your environment (e.g. blocked host), download the RGB zip manually from the
Kaggle link in the task brief and point `DATA_ROOT` at the extracted `EuroSAT` folder instead — the rest
of the notebook doesn't change.


In [ ]:
DATA_ROOT = "./data"
IMG_SIZE = 64          # EuroSAT images are natively 64x64
BATCH_SIZE = 64
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Base transform used for the "no augmentation" runs (still needs resize/tensor/normalize —
# that's preprocessing, not augmentation).
NORM_MEAN = [0.485, 0.456, 0.406]   # ImageNet stats, needed for the pretrained-model branch too
NORM_STD  = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# Augmented transform for the "with augmentation" runs.
aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# Load once with a plain transform just to get the split indices, then wrap with the transform we need.
full_dataset = datasets.EuroSAT(root=DATA_ROOT, download=True, transform=base_transform)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}):", CLASS_NAMES)
print("Total images:", len(full_dataset))

n_total = len(full_dataset)
n_val = int(VAL_SPLIT * n_total)
n_test = int(TEST_SPLIT * n_total)
n_train = n_total - n_val - n_test

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset, test_subset = random_split(
    full_dataset, [n_train, n_val, n_test], generator=generator
)
print(f"Train / Val / Test sizes: {n_train} / {n_val} / {n_test}")


In [ ]:
class TransformedSubset(torch.utils.data.Dataset):
    """Wraps a Subset so we can apply a different transform (e.g. augmentation) to the
    same underlying indices without re-downloading or re-splitting the dataset."""
    def __init__(self, subset, base_ds_root, transform):
        self.subset = subset
        self.transform = transform
        self.underlying = datasets.EuroSAT(root=base_ds_root, download=False, transform=transform)

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        real_idx = self.subset.indices[idx]
        return self.underlying[real_idx]


def make_loaders(transform_train):
    """Builds train/val/test DataLoaders. Val/test always use base_transform (no augmentation) —
    augmentation is a training-time regularizer, not something you evaluate through."""
    train_ds = TransformedSubset(train_subset, DATA_ROOT, transform_train)
    val_ds   = TransformedSubset(val_subset, DATA_ROOT, base_transform)
    test_ds  = TransformedSubset(test_subset, DATA_ROOT, base_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader

# Loaders for the two data regimes
train_loader_noaug, val_loader, test_loader = make_loaders(base_transform)
train_loader_aug, _, _ = make_loaders(aug_transform)   # val/test loaders identical, reuse the ones above


## 2. Model 1 — TinyVGG (trained from scratch)

A small VGG-style CNN: two conv blocks (Conv → ReLU → Conv → ReLU → MaxPool) followed by a classifier head.
This is trained from random initialization — no pretrained weights.

In [ ]:
class TinyVGG(nn.Module):
    def __init__(self, in_channels=3, hidden_units=32, num_classes=NUM_CLASSES, img_size=IMG_SIZE):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units * 2, hidden_units * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        flat_size = (hidden_units * 2) * (img_size // 4) * (img_size // 4)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

tiny_vgg_test = TinyVGG().to(DEVICE)
print(tiny_vgg_test)


## 3. Shared training / evaluation utilities

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr=1e-3, label=""):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss, running_correct, n_seen = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            running_correct += (outputs.argmax(1) == labels).sum().item()
            n_seen += images.size(0)

        train_loss = running_loss / n_seen
        train_acc = running_correct / n_seen

        model.eval()
        val_loss, val_correct, val_seen = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_seen += images.size(0)

        val_loss /= val_seen
        val_acc = val_correct / val_seen

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[{label}] Epoch {epoch+1}/{epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    return history


@torch.no_grad()
def evaluate_on_test(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, seen = 0.0, 0, 0

    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        seen += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    test_loss = total_loss / seen
    test_acc = correct / seen
    return test_loss, test_acc, all_labels, all_preds


def plot_curves(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} — Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_confusion(all_labels, all_preds, title):
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix — {title}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    return cm


## 4. Train Model 1 — TinyVGG, no augmentation

In [ ]:
EPOCHS = 10

tinyvgg_noaug = TinyVGG().to(DEVICE)
hist_tinyvgg_noaug = train_model(
    tinyvgg_noaug, train_loader_noaug, val_loader, epochs=EPOCHS, label="TinyVGG-NoAug"
)


In [ ]:
plot_curves(hist_tinyvgg_noaug, "TinyVGG (No Augmentation)")
test_loss_1, test_acc_1, labels_1, preds_1 = evaluate_on_test(tinyvgg_noaug, test_loader)
print(f"TinyVGG (No Aug) — Test loss: {test_loss_1:.4f}, Test acc: {test_acc_1:.4f}")
cm_1 = plot_confusion(labels_1, preds_1, "TinyVGG (No Aug)")
print(classification_report(labels_1, preds_1, target_names=CLASS_NAMES))


## 5. Train Model 2 — TinyVGG, with augmentation

In [ ]:
tinyvgg_aug = TinyVGG().to(DEVICE)
hist_tinyvgg_aug = train_model(
    tinyvgg_aug, train_loader_aug, val_loader, epochs=EPOCHS, label="TinyVGG-Aug"
)


In [ ]:
plot_curves(hist_tinyvgg_aug, "TinyVGG (With Augmentation)")
test_loss_2, test_acc_2, labels_2, preds_2 = evaluate_on_test(tinyvgg_aug, test_loader)
print(f"TinyVGG (Aug) — Test loss: {test_loss_2:.4f}, Test acc: {test_acc_2:.4f}")
cm_2 = plot_confusion(labels_2, preds_2, "TinyVGG (Aug)")
print(classification_report(labels_2, preds_2, target_names=CLASS_NAMES))


## 6. Model 3 & 4 — Fine-tuned pretrained model (ResNet18)

We "retrofit an off-the-shelf sensor": load ImageNet-pretrained ResNet18, freeze the convolutional
backbone, and replace + train only the final classifier layer (plus optionally unfreeze `layer4` for a
light fine-tune). This is much faster to converge than training from scratch.

In [ ]:
def build_pretrained_model(num_classes=NUM_CLASSES, unfreeze_layer4=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    if unfreeze_layer4:
        for param in model.layer4.parameters():
            param.requires_grad = True

    # Replace the classifier head — this is always trainable
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model


### 6a. Fine-tune — no augmentation

In [ ]:
resnet_noaug = build_pretrained_model().to(DEVICE)
hist_resnet_noaug = train_model(
    resnet_noaug, train_loader_noaug, val_loader, epochs=EPOCHS, lr=1e-4, label="ResNet18-NoAug"
)


In [ ]:
plot_curves(hist_resnet_noaug, "ResNet18 Fine-tuned (No Augmentation)")
test_loss_3, test_acc_3, labels_3, preds_3 = evaluate_on_test(resnet_noaug, test_loader)
print(f"ResNet18 (No Aug) — Test loss: {test_loss_3:.4f}, Test acc: {test_acc_3:.4f}")
cm_3 = plot_confusion(labels_3, preds_3, "ResNet18 (No Aug)")
print(classification_report(labels_3, preds_3, target_names=CLASS_NAMES))


### 6b. Fine-tune — with augmentation

In [ ]:
resnet_aug = build_pretrained_model().to(DEVICE)
hist_resnet_aug = train_model(
    resnet_aug, train_loader_aug, val_loader, epochs=EPOCHS, lr=1e-4, label="ResNet18-Aug"
)


In [ ]:
plot_curves(hist_resnet_aug, "ResNet18 Fine-tuned (With Augmentation)")
test_loss_4, test_acc_4, labels_4, preds_4 = evaluate_on_test(resnet_aug, test_loader)
print(f"ResNet18 (Aug) — Test loss: {test_loss_4:.4f}, Test acc: {test_acc_4:.4f}")
cm_4 = plot_confusion(labels_4, preds_4, "ResNet18 (Aug)")
print(classification_report(labels_4, preds_4, target_names=CLASS_NAMES))


## 7. Final comparison — the Project Sentinel field report

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Model": ["TinyVGG (scratch)", "TinyVGG (scratch)", "ResNet18 (fine-tuned)", "ResNet18 (fine-tuned)"],
    "Augmentation": ["No", "Yes", "No", "Yes"],
    "Test Loss": [test_loss_1, test_loss_2, test_loss_3, test_loss_4],
    "Test Accuracy": [test_acc_1, test_acc_2, test_acc_3, test_acc_4],
})
summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
labels_x = ["TinyVGG\nNo Aug", "TinyVGG\nAug", "ResNet18\nNo Aug", "ResNet18\nAug"]
accs = [test_acc_1, test_acc_2, test_acc_3, test_acc_4]
bars = ax.bar(labels_x, accs, color=["#4C72B0", "#4C72B0", "#DD8452", "#DD8452"])
ax.set_ylabel("Test Accuracy")
ax.set_title("Final Test Accuracy — All Four Configurations")
ax.set_ylim(0, 1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01, f"{acc:.3f}", ha="center")
plt.tight_layout()
plt.show()


### Field report — write-up

Fill this in after running the cells above with your actual numbers.

**1. Scratch CNN vs. fine-tuned model**
- Which converged faster (fewer epochs to plateau)? Why — think about what the pretrained backbone
  already "knows" (edges, textures) vs. what TinyVGG has to learn from zero.
- Which reached higher final test accuracy?

**2. Effect of augmentation**
- Did augmentation help TinyVGG more or less than it helped ResNet18? Augmentation mainly fights
  overfitting — a model with less capacity to overfit (or one already regularized by frozen pretrained
  weights) may benefit less.
- Look at the train/val gap in the loss curves: did augmentation shrink it?

**3. Confusion matrix insights**
- Which land-cover classes get confused with each other across all four models (e.g. Highway ↔
  Industrial, or PermanentCrop ↔ AnnualCrop)? These visually-similar classes are the ones a real
  reserve-monitoring pipeline would need extra care with.

**4. Deployment recommendation for Project Sentinel**
- Given the satellite pipeline needs to tag every few hours: which of the four models would you actually
  ship, trading off accuracy vs. training/inference cost? Justify in 2-3 sentences.
